# Tool Design and MCP integration


Your MCP tool for processing customer refunds is called by multiple different agents. How should you configure access to prevent a reporting agent from accidentally triggering a refund?
A.Add authorization checks inside the `process_refund` tool itself
B.Scope tool access: only include `process_refund` in the allowedTools of agents that need it; exclude it from reporting and read-only agents
C.Add a system prompt instruction telling the reporting agent not to use `process_refund`
D.Use `tool_choice: "auto"` which will prevent accidental calls
✓ Correct — Explanation
Tool access should be scoped to each agent's role. A reporting agent should never have `process_refund` in its `allowedTools` — if the tool isn't available, it literally cannot be called. Prompt instructions and `tool_choice: "auto"` are probabilistic guardrails that can still fail. Defense-in-depth starts with not giving agents tools they should never use.



What does the MCP `isError` flag on a tool response indicate?
A.The tool encountered a runtime error and the agent should handle the failure
B.The tool call was rejected by the MCP server before execution even started due to invalid params
C.Claude made an invalid tool call with wrong parameter types or values
D.The tool result should be ignored and the agent should proceed without it
✓ Correct — Explanation
`isError: true` on an MCP tool response means the tool was called successfully (no schema error) but encountered a runtime error during execution. The agent should read the error metadata and decide whether to retry, escalate, or continue with partial data.


A developer needs to find all files in a repository that have the extension .test.tsx. Which built-in tool should they use?
A.Grep with the pattern ".test.tsx$" to search for matching filenames
B.Read with a directory path argument to recursively list all contents
C.Glob with the pattern "**/*.test.tsx" to match files by extension
D.Bash with the find command to locate files matching the extension
✓ Correct — Explanation
Glob is the built-in tool for finding files by name or extension patterns. The pattern **/*.test.tsx matches all files ending in .test.tsx anywhere in the directory tree. Grep searches file contents, not file names — it would need the -l flag and wouldn't efficiently match by extension. Read operates on individual files or directories but doesn't support recursive pattern matching. Bash with find works but bypasses the purpose-built Glob tool.


During testing, you observe that the synthesis agent frequently needs to verify specific claims while combining findings. Currently, when verification is needed, the synthesis agent returns control to the coordinator, which invokes the web search agent, then re-invokes synthesis with results. This adds 2–3 round trips per task and increases latency by 40%. Your evaluation shows that 85% of these verifications are simple fact-checks (dates, names, statistics) while 15% require deeper investigation. What's the most effective approach to reduce overhead while maintaining system reliability?
A.Give the synthesis agent a scoped verify_fact tool for simple lookups, while complex verifications continue delegating to the web search agent through the coordinator
B.Have the synthesis agent accumulate all verification needs and return them as a batch to the coordinator at the end of its pass, which then sends them all to the web search agent at once
C.Give the synthesis agent access to all web search tools so it can handle any verification need directly without round-trips through the coordinator
D.Have the web search agent proactively cache extra context around each source during initial research, anticipating what the synthesis agent might need to verify
✗ Incorrect — Explanation
Option A applies the principle of least privilege by giving the synthesis agent only what it needs for the 85% common case (simple fact verification) while preserving the existing coordination pattern for complex cases. Option B's batching approach creates blocking dependencies since synthesis steps may depend on earlier verified facts. Option C over-provisions the synthesis agent, violating separation of concerns. Option D relies on speculative caching that cannot reliably predict what the synthesis agent will need to verify.


An MCP tool returns `{"isError": false, "content": []}` when a customer lookup finds no matching records. A different call returns `{"isError": true, "errorCategory": "permission"}` when the agent lacks database access. Why is this distinction important?
A.Both should return `isError: true` for consistent unified error handling across all tool responses
B.The `isError` flag is optional and both responses need the same handling
C.Empty results are a successful query; permission errors need different recovery
D.Empty results and permission errors should be treated identically as no-data
✓ Correct — Explanation
This distinction is critical for agent recovery logic. `isError: false` + empty content = the query ran successfully, no records exist — do not retry. `isError: true` + permission error = the agent doesn't have access — retry won't help, escalation or credential rotation is needed. Treating them identically causes agents to retry successful queries or give up on fixable access errors.


What is the primary mechanism Claude uses to decide which tool to invoke?
A.The tool name only — Claude does not read or parse the tool description text at all
B.The tool description, which explains purpose, expected inputs, and boundaries
C.The order in which tools appear in the tools array sent to the API
D.A separate routing system prompt that maps user intents to tool names
✗ Incorrect — Explanation
Tool descriptions are the primary selection mechanism. Claude reads each tool's description to determine which tool is appropriate for the current task. Weak, minimal, or overlapping descriptions directly cause misrouting. This is why investing in clear, differentiated descriptions is critical for reliable tool use.


An MCP server exposes 3 tools: `search_knowledge_base`, `search_web`, and `search_database`. All three have the description: "Searches for information." A system prompt says "Search for relevant information before answering." What two changes are needed?
A.Rename the tools to remove the word "search" and update the system prompt
B.Rewrite each tool description to explicitly distinguish its source, input format, and when to use it vs. the others; and add a system prompt instruction that maps specific query types to the correct tool
C.Merge all three into a single `search` tool with a `source` parameter
D.Add few-shot examples showing the correct tool for different query types
✓ Correct — Explanation
Two changes needed: (1) Descriptions must differentiate: `search_knowledge_base` — "search internal company documentation; use when the question is about company policies, products, or procedures; input: keyword query; output: internal doc excerpts with IDs"; `search_web` — "search public internet; use for current events, external facts, recent information"; `search_database` — "query structured customer/transaction data; use when needing specific records by ID or date range." (2) System prompt should map task types to appropriate tools to reinforce the descriptions.


Production logs show the agent sometimes selects get_customer when lookup_order would be more appropriate, particularly for ambiguous requests like "I need help with my recent purchase." You decide to add few-shot examples to your system prompt to improve tool selection. Which approach will most effectively address this issue?
A.Add examples grouped by tool—all get_customer scenarios together, then all lookup_order scenarios.
B.Add 10–15 examples of clear, unambiguous requests that demonstrate correct tool selection for each tool's typical use cases.
C.Add explicit "use when" and "do not use when" guidelines in each tool's description covering the ambiguous cases.
D.Add 4–6 examples targeting ambiguous scenarios, each showing reasoning for why one tool was chosen over plausible alternatives.
✓ Correct — Explanation
The error occurs on ambiguous requests — not on clear cases where the agent already performs correctly. Few-shot examples are most effective when they target the specific scenarios where errors occur, paired with explicit reasoning about the comparative decision. For "I need help with my recent purchase," the reasoning might be: "This mentions a purchase, not account details — use lookup_order, not get_customer. If the customer had said 'my account' or 'my profile,' get_customer would be appropriate." This comparative reasoning directly teaches the decision process for edge cases. Grouping by tool (A) makes examples easier to scan but does not demonstrate comparative reasoning. Many clear-case examples (B) reinforce behavior that already works correctly without addressing the ambiguous cases. Tool description updates (C) are a valid complementary fix but are not few-shot examples.


Your team wants to share a GitHub MCP server configuration so all developers have access to it when they clone the repository. Where should you configure this MCP server?
A.In the project's .mcp.json file, committed to version control for the team
B.In ~/.claude.json on each developer's machine for personal MCP server configuration
C.In the root CLAUDE.md file as a structured MCP configuration code block
D.In a .env file at the project root with MCP_SERVER_URL variables defined
✗ Incorrect — Explanation
Project-level MCP server configuration belongs in .mcp.json at the project root. This file is committed to version control and automatically available to all team members when they clone or pull the repository. ~/.claude.json is for personal or experimental MCP servers that should not be shared with the team. CLAUDE.md is for instructions and context, not server configuration. A .env file doesn't configure MCP servers.


What are the three values available for the `tool_choice` parameter?
A."always", "never", "auto" with an optional tool name override parameter
B."required", "optional", "forced" as the three selection modes
C."auto", "required", "none" matching the standard API values
D."auto", "any", and a forced object `{"type": "tool", "name": "..."}`
✗ Incorrect — Explanation
`tool_choice` options: `"auto"` (Claude decides whether to use a tool), `"any"` (Claude must use at least one tool), and `{"type": "tool", "name": "tool_name"}` (Claude must call this specific named tool). There is no "required" or "none" option.



Scenario: Customer Support Resolution Agent
Your support agent has four MCP tools: `get_customer`, `lookup_order`, `process_refund`, and `escalate_to_human`. Over time, `get_customer` is being called for order-related queries and `lookup_order` is being called to fetch customer profiles. The tool descriptions are: `get_customer`: "Gets customer information" and `lookup_order`: "Gets order information." What is the correct fix?
A.Rename the tools to `customer_tool` and `order_tool` for clarity
B.Rewrite both descriptions to explicitly state their purpose, expected inputs (customer ID vs order ID), returned fields, and when to use each vs alternatives
C.Add a routing system prompt that maps keywords to tools
D.Merge both tools into a single `get_record` tool with a `type` parameter
✓ Correct — Explanation
One-line generic descriptions like "Gets customer information" are the most common cause of tool misrouting. Each description should include: (1) exactly what it fetches, (2) the required input format (e.g., "Takes a `customer_id` string — use this when you have a customer ID, not an order ID"), (3) what it returns, and (4) when NOT to use it. Detailed descriptions are the primary routing mechanism Claude has available.


You add a powerful `semantic_search_codebase` MCP tool that outperforms Grep for conceptual queries across a large monorepo. However, you observe that Claude consistently uses Grep instead. What is the most likely cause and fix?
A.The tool description is too vague — rewrite it to explain when it outperforms Grep
B.The MCP server has higher latency than Grep; reduce server response time to make it competitive
C.Claude always prefers built-in tools over MCP tools; this cannot be changed
D.Add a system prompt instruction: "always use semantic_search_codebase instead"
✓ Correct — Explanation
Claude selects tools based on their descriptions. If `semantic_search_codebase` has a generic description like "search the codebase", Claude has no basis for preferring it over Grep, which it already understands well. The fix is to write a description that explains the tool's unique capabilities: "Use this tool for conceptual or semantic queries where you need to find code related to a concept, pattern, or intent — not just a literal string. More accurate than Grep for queries like 'authentication middleware' or 'payment processing logic'." Latency (A) is not the cause here. Claude does not unconditionally prefer built-in tools (B). A system prompt instruction (D) is fragile compared to a well-written description.

The support agent calls the `lookup_order` tool and receives: `{"isError": false, "content": [], "message": "No orders found for customer ID 48291"}`. It then calls `lookup_order` again three more times with the same customer ID. What is wrong with the agent's behavior and what caused it?
A.The agent is retrying correctly — transient errors should be retried
B.The agent is treating a valid empty result (no orders found) as an error, causing unnecessary retries. The tool should have returned `isError: false` with an empty array — which it did — and the agent should have accepted this as a successful lookup with no results
C.The `lookup_order` tool should have returned `isError: true` to prevent this confusion
D.The agent needs better retry logic with exponential back-off
✓ Correct — Explanation
The tool returned `isError: false` with an empty array — this is correct behavior for "no records found." The agent erroneously interpreted this as a failure and retried. The root cause is missing downstream logic to distinguish empty results (success, no data) from errors (failure, try again). This is the critical `isError: false + empty content` vs `isError: true` distinction the exam tests directly.


An agent attempts to use the Edit tool to update a configuration value in a large auto-generated file where the same string "timeout: 30" appears 47 times. The Edit call fails because the match is not unique. What is the correct fallback approach?
A.Use Grep to find the correct line number, then pass it as an Edit offset
B.Use Bash with sed to replace the specific occurrence by its line number
C.Use Edit with a larger surrounding context block to make the old_string match unique
D.Use Read to load the full file, make the change, then Write the result back
✗ Incorrect — Explanation
When Edit fails due to non-unique text matches, the correct fallback is Read + Write: load the full file with Read, apply the targeted modification programmatically, and save the result with Write. This provides reliable modification even when no unique anchor text exists. Option D (larger context) would work if unique context exists, but when the same pattern repeats 47 times throughout an auto-generated file, finding a unique anchor may be impossible. Bash sed (C) works but is fragile and bypasses the purpose-built tools. Grep + Edit offset (A) doesn't exist — Edit doesn't accept line-number offsets.


You want to validate all tool inputs against a strict schema and ensure Claude never passes missing or incorrectly typed parameters. What configuration achieves this?
A.Add a `validate: true` flag to the tool definition
B.Use `strict: true` in tool definitions to enable Structured Outputs for guaranteed schema validation
C.Write a PostToolUse hook that validates inputs after the call
D.Add explicit validation instructions to the tool description
✓ Correct — Explanation
`strict: true` in tool definitions enables Structured Outputs mode, which guarantees that Claude's tool calls always match the defined schema exactly — no missing required fields, no type mismatches. This eliminates an entire class of production failures where Claude generates syntactically valid but schema-invalid tool calls.


You have a generic `analyze_document` tool that is being called for web results, PDFs, and database records with inconsistent quality. What is the best refactoring approach?
A.Add a `document_type` parameter to the existing tool and update the description
B.Split into purpose-specific tools: `extract_web_results`, `parse_pdf_content`, `query_database_record` — each with defined input/output contracts
C.Add more few-shot examples showing correct document type handling
D.Create a pre-routing tool that classifies the document type before calling `analyze_document`
✗ Incorrect — Explanation
Generic multi-purpose tools with parameter-based branching are harder for Claude to route correctly than purpose-specific tools with clear descriptions. Splitting into `extract_web_results` (web content, returns structured web data), `parse_pdf_content` (PDFs, returns sections + metadata), and `query_database_record` (structured queries, returns typed fields) gives Claude distinct routing signals and produces more reliable, higher-quality outputs.



A web research subagent and a document analysis subagent both have a tool called `extract_information`. The coordinator occasionally routes document analysis requests to the wrong subagent. What is causing this and how should it be fixed?
A.The coordinator needs more context about which subagent to use — add routing instructions to the coordinator system prompt
B.Rename both tools with role-specific names and update descriptions to eliminate overlap: e.g., `extract_web_search_results` vs `extract_document_sections`
C.Merge the two subagents into one to eliminate ambiguity
D.Add a `target_subagent` parameter to the Task tool call
✓ Correct — Explanation
Identical tool names across subagents create routing ambiguity at the coordinator level. The fix is to give each subagent's tools distinct, role-specific names that make the boundary obvious: `extract_web_search_results` (web subagent) vs `extract_document_sections` (document subagent). The coordinator can now route unambiguously based on the task type.


Your team needs to integrate Claude Code with Jira to let agents create and update tickets. A developer proposes building a custom MCP server for Jira. What is the recommended approach?
A.Build a custom MCP server to have full control over the Jira API surface and schema exposed
B.Connect directly to the Jira REST API via a generic `fetch_url` tool instead
C.Embed Jira credentials in the system prompt and use built-in web fetch tool
D.Use the existing community MCP server; reserve custom builds for unique needs
✓ Correct — Explanation
For standard integrations like Jira, GitHub, or Slack, the MCP ecosystem has community servers that are already built, tested, and maintained. Using an existing community server eliminates engineering overhead and gets the integration working immediately. Custom MCP servers should be reserved for workflows that are genuinely team-specific — internal databases, proprietary APIs, or unusual tool compositions that have no existing solution. A generic `fetch_url` tool (C) bypasses the structured, purpose-built capabilities of an MCP server. Embedding credentials in the system prompt (D) is a security anti-pattern.


A synthesis subagent is given access to 18 tools: web search, document analysis, database query, file read/write, email, calendar, and 12 others. What is the most likely consequence?
A.The subagent will use all 18 tools for every task, slowing performance down
B.The MCP server will reject the connection due to the hard tool count limit
C.The synthesis subagent will request a tool manifest from the coordinator agent
D.Tool selection degrades — Claude makes worse decisions with too many options
✓ Correct — Explanation
Tool selection reliability degrades as the number of available tools increases, especially when many tools are outside the agent's core role. A synthesis agent with 18 tools will sometimes misuse web search or email tools. The principle is to scope each agent's tools to 4–5 role-relevant tools, with limited cross-role tools only for high-frequency needs.


You are configuring a GitHub MCP server in your project's .mcp.json. The server requires a personal access token. A teammate suggests hardcoding the token directly in the JSON file since it's an internal repo. What is the correct approach?
A.Hardcode the token — internal repos don't need the same security standards
B.Use ${GITHUB_TOKEN} env var expansion in .mcp.json; each dev sets it locally
C.Store the token in CLAUDE.md under a dedicated secrets configuration section
D.Create a .mcp.local.json file with the token and add it to the .gitignore
✓ Correct — Explanation
.mcp.json supports environment variable expansion using ${VAR_NAME} syntax. This allows the shared configuration file to be committed safely without containing credentials — each developer sets the actual token value in their shell environment (e.g., ~/.zshrc or a local .env). Hardcoding tokens (A) is a security anti-pattern even in internal repos — tokens get leaked through git history, screenshots, and team changes. CLAUDE.md is not a secrets store. A .mcp.local.json approach isn't a supported pattern.


Production logs show the agent frequently calls get_customer when users ask about orders (e.g., "check my order #12345"), instead of calling lookup_order. Both tools have minimal descriptions ("Retrieves customer information" / "Retrieves order details") and accept similar identifier formats. What's the most effective first step to improve tool selection reliability?
A.Add few-shot examples to the system prompt demonstrating correct tool selection patterns, with 5–8 examples showing order-related queries routing to lookup_order
B.Expand each tool's description to include input formats it handles, example queries, edge cases, and boundaries explaining when to use it versus similar tools
C.Implement a routing layer that parses user input before each turn and pre-selects the appropriate tool based on detected keywords and identifier patterns
D.Consolidate both tools into a single lookup_entity tool that accepts any identifier and internally determines which backend to query
✓ Correct — Explanation
Tool descriptions are the primary mechanism LLMs use for tool selection. When descriptions are minimal, models lack the context to differentiate between similar tools. Option B directly addresses this root cause with a low-effort, high-leverage fix. Few-shot examples (A) add token overhead without fixing the underlying issue. A routing layer (C) is over-engineered and bypasses the LLM's natural language understanding. Consolidating tools (D) is a valid architectural choice but requires more effort than a "first step" warrants when the immediate problem is inadequate descriptions.



You want to ensure Claude always calls the `extract_structured_data` tool for a specific step in a pipeline, regardless of what the conversation context suggests. Which `tool_choice` configuration achieves this?
A.`tool_choice: {"type": "tool", "name": "extract_structured_data"}`
B.`tool_choice: "auto"` which lets Claude decide based on conversation context
C.`tool_choice: "any"` which guarantees some tool call but not a specific one
D.`tool_choice: "required"` with the tool name passed in a separate parameter
✓ Correct — Explanation
Forced tool selection requires `{"type": "tool", "name": "tool_name"}`. This guarantees Claude calls that specific tool regardless of context. `"auto"` lets Claude decide freely. `"any"` requires *some* tool call but not a specific one. `"required"` is not a valid value.


An MCP tool encounters a permanent business rule violation (refund exceeds the allowed maximum). Which `isRetryable` value should the error response include?
A.true
B.null
C.false
D.The field should be omitted for business errors
✗ Incorrect — Explanation
Business rule violations are not retryable — retrying with the same parameters will always produce the same error. Setting `isRetryable: false` signals this to the agent, preventing wasted retry attempts. Transient errors (timeouts, temporary service unavailability) should use `isRetryable: true`.


You have two tools: `analyze_content` (description: "Analyzes content") and `analyze_document` (description: "Analyzes a document"). What problem will this cause in production?
A.Both tools will be called simultaneously for every request, causing duplicate data processing
B.Unreliable tool selection — Claude misroutes because descriptions are identical
C.Claude will refuse to use either tool due to the ambiguity in descriptions
D.The tools will merge into a single logical tool inside the agent runtime
✓ Correct — Explanation
Identical or near-identical descriptions make it impossible for Claude to reliably distinguish between tools. With no functional differentiation, Claude will select unpredictably. The fix is to make each tool's description explicitly explain: (1) what it does differently, (2) its specific input format, (3) when to use it instead of similar alternatives.


Your extraction system calls an `extract_fields` MCP tool on a 500-document batch. For 12% of documents, the tool returns `{"isError": true, "errorCategory": "validation", "isRetryable": false, "message": "Document format not supported: .pages"}`. How should the system handle these?
A.Retry each failed document three times before skipping
B.Skip all failed documents without logging
C.Since `isRetryable: false` and the error is a validation/format issue, log the specific files with their error details, continue processing the remaining documents, and report the failure summary to the operator
D.Halt the entire batch and fix the format issue before restarting
✓ Correct — Explanation
`isRetryable: false` means retrying will not help — the format is unsupported and retrying the same document will always fail. The correct response is to: (1) skip these documents without retry, (2) log each failure with the document name and error details for operator review, (3) continue processing supported formats, (4) include a failure summary in the final report. Halting the entire batch is disproportionate when most documents are processable.


An MCP tool connects to a customer database and returns `{"isError": true, "errorCategory": "transient", "isRetryable": true, "message": "Connection timeout after 30s"}`. The agent is a subagent in a coordinator-subagent system. What should happen next?
A.The subagent should immediately propagate the error to the coordinator
B.The subagent should attempt local retry logic for the transient error; propagate to coordinator only if retries are exhausted, including partial results and what was attempted
C.The subagent should return empty results and continue
D.The coordinator should restart the entire pipeline from scratch
✓ Correct — Explanation
Transient errors (`isRetryable: true`) should be handled locally by the subagent first — retry with appropriate back-off. Only when local recovery fails should the error propagate to the coordinator, along with: (1) what data was retrieved before the failure, (2) the specific error details, (3) what was attempted, (4) whether further retries are advisable. This preserves partial work and enables informed coordinator decisions.


Your developer productivity agent uses built-in tools (Read, Write, Bash, Grep, Glob) plus 14 custom MCP tools for linting, type-checking, test running, deployment, database migration, and more. Engineers report the agent is using deployment tools during routine code exploration tasks. What is the architectural fix?
A.Add system prompt instructions: "Do not use deployment tools during exploration"
B.Reduce the total tool count to 5 by merging related tools
C.Create task-scoped tool profiles: exploration tasks get Read/Grep/Glob only; development tasks add Write/Bash/Lint/TypeCheck; deployment tasks add the deployment/migration tools
D.Move deployment tools to a separate MCP server
✓ Correct — Explanation
The solution is task-scoped tool profiles — dynamically configure `allowedTools` based on the current task phase. Exploration: read-only tools only. Development: add write and quality tools. Deployment: add deployment tools explicitly. This prevents accidental deployment tool invocations by making them literally unavailable during exploration, rather than relying on prompt instructions that can fail.






